In [5]:
## Script para consumir dados da API e enviar para o banco de dados.
#Importação de bibliotecas
import pandas as pd
from src.database.banco import conectar, conectar_alchemy
from sqlalchemy import create_engine
from jupyter_client.localinterfaces import PUBLIC_IPS
import urllib
#Criação da conexão como sql alchemy
engine = conectar_alchemy()
# Leitura do arquivo CSV baixado no Kaggle
df = pd.read_csv(
    r"src/data/books.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    low_memory=False,
)
#Comandos para melhor visualização das colunas e linhas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
#Cópia para tratamento de erros e versão final
df_final = df[["Book-Title","Book-Author","Year-Of-Publication", "Publisher"]].copy()
#Tratamento de erros
df_final.rename(columns={"Book-Title":"nome", "Book-Author":"autor","Year-Of-Publication":"ano","Publisher":"editora"}, inplace=True)
df_final.dropna(inplace=True)
linhas_com_erro = df_final[df_final["ano"].str.isnumeric() == False].index
nome_autor = df_final.loc[linhas_com_erro, "nome"].str.split('\";', expand=True)
nome_autor_formatado = nome_autor[1].str.strip('"')
nome_nome_formatado = nome_autor[0].str.strip(r'\"')
df_final.loc[linhas_com_erro, "editora"] = df_final.loc[linhas_com_erro, "ano"]
df_final.loc[linhas_com_erro, "ano"] = df_final.loc[linhas_com_erro, "autor"]
df_final.loc[linhas_com_erro, "autor"] = nome_autor_formatado
df_final.loc[linhas_com_erro, "nome"] = nome_nome_formatado
df_final['autor'] = df_final['autor'].str.replace("ClÃ?Â©zio", "Clézio")
df_final["ano"] = df_final["ano"].astype(int)
lista_caracteres = r'%|¨|@'
colunas_para_checagem = ["nome", "autor","editora"]
colunas_com_utf = pd.DataFrame()
colunas_com_erro = pd.DataFrame()
for coluna in colunas_para_checagem:
    filtro_encode = df_final[coluna].str.encode('latin1').str.decode('utf-8', errors='ignore')
    colunas_com_utf[coluna] = filtro_encode
    filtro_restante = colunas_com_utf[colunas_com_utf[coluna].str.contains(lista_caracteres,na=False)]
    colunas_com_erro = pd.concat([colunas_com_erro,filtro_restante])

df_final = df_final.drop(index=colunas_com_erro.index)
df_final = df_final.reset_index(drop=True)
#Comando para transformar o dataframe em SQL
df_final.to_sql(
    name="Livros",
    con=engine,
    if_exists="append",
    index=False,
)
#Print para acompanhar o andamento do script
print("Injeção concluída! Todos os dados foram enviados.")
#Encerrando conexão
engine.dispose()





Injeção concluída! Todos os dados foram enviados.
